In [ ]:
!pip install numpy pandas matplotlib --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)


In [ ]:
def build_shoe(n_decks=8):
    """8-deck shoe. Card values: A=11 (adjusted for soft hands), 2-9=face, 10/J/Q/K=10."""
    single_deck = [11, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10]
    deck = single_deck * 4
    shoe = deck * n_decks
    np.random.shuffle(shoe)
    return shoe

def hand_value(cards):
    """Returns (best_total, is_soft). Soft = an Ace is currently counted as 11."""
    total = sum(cards)
    aces = cards.count(11)
    while total > 21 and aces > 0:
        total -= 10
        aces -= 1
    soft = (11 in cards) and (aces > 0) and (total <= 21)
    return total, soft

def is_blackjack(cards):
    return len(cards) == 2 and hand_value(cards)[0] == 21




In [ ]:
def play_dealer(shoe, pos, dealer_cards):
    total, soft = hand_value(dealer_cards)
    while total < 17:
        dealer_cards.append(shoe[pos])
        pos += 1
        total, soft = hand_value(dealer_cards)
    return dealer_cards, pos


In [ ]:
def basic_strategy(player_cards, dealer_upcard, can_split=True, can_double=True):
    total, soft = hand_value(player_cards)

    if can_split and len(player_cards) == 2 and player_cards[0] == player_cards[1]:
        pair_val = player_cards[0]
        if pair_val == 11:
            return 'P'
        if pair_val == 10:
            return 'S'
        if pair_val == 9:
            return 'P' if dealer_upcard not in [7, 10, 11] else 'S'
        if pair_val == 8:
            return 'P'
        if pair_val == 7:
            return 'P' if dealer_upcard <= 7 else 'H'
        if pair_val in [2, 3]:
            return 'P' if dealer_upcard <= 7 else 'H'
        if pair_val == 6:
            return 'P' if dealer_upcard <= 6 else 'H'
        if pair_val == 5:
            return 'D' if (can_double and dealer_upcard <= 9) else 'H'
        if pair_val == 4:
            return 'P' if dealer_upcard in [5, 6] else 'H'

    if soft:
        if total >= 19:
            return 'S'
        if total == 18:
            if dealer_upcard in [2, 3, 4, 5, 6]:
                return 'D' if can_double else 'S'
            return 'S' if dealer_upcard in [7, 8] else 'H'
        if total == 17:
            return 'D' if (can_double and dealer_upcard in [3, 4, 5, 6]) else 'H'
        if total in [15, 16]:
            return 'D' if (can_double and dealer_upcard in [4, 5, 6]) else 'H'
        if total in [13, 14]:
            return 'D' if (can_double and dealer_upcard in [5, 6]) else 'H'
        return 'H'

    if total >= 17:
        return 'S'
    if total >= 13:
        return 'S' if dealer_upcard <= 6 else 'H'
    if total == 12:
        return 'S' if dealer_upcard in [4, 5, 6] else 'H'
    if total == 11:
        return 'D' if can_double else 'H'
    if total == 10:
        return 'D' if (can_double and dealer_upcard <= 9) else 'H'
    if total == 9:
        return 'D' if (can_double and dealer_upcard in [3, 4, 5, 6]) else 'H'
    return 'H'


In [ ]:
def play_out_hand(shoe, pos, cards, dealer_upcard, is_split_aces=False):
    """Plays one hand until stand/bust/double. Returns (cards, bet_multiplier, pos)."""
    bet_multiplier = 1

    if is_split_aces:
        # Standard casino rule: split Aces get exactly one card each, no further hits
        cards.append(shoe[pos])
        pos += 1
        return cards, bet_multiplier, pos

    while True:
        can_double = (len(cards) == 2)
        action = basic_strategy(cards, dealer_upcard, can_split=False, can_double=can_double)

        if action == 'D' and can_double:
            bet_multiplier = 2
            cards.append(shoe[pos])
            pos += 1
            break
        elif action in ('H', 'D'):
            cards.append(shoe[pos])
            pos += 1
            total, _ = hand_value(cards)
            if total > 21:
                break
        else:
            break

    return cards, bet_multiplier, pos


In [ ]:
def play_hand(shoe, pos):
    """Returns (list_of_(result, bet_multiplier), cards_used). More than one result if split."""
    start_pos = pos
    player = [shoe[pos], shoe[pos + 2]]
    dealer = [shoe[pos + 1], shoe[pos + 3]]
    pos += 4
    dealer_upcard = dealer[0]

    player_bj = is_blackjack(player)
    dealer_bj = is_blackjack(dealer)

    if player_bj or dealer_bj:
        if player_bj and dealer_bj:
            return [('push', 1)], pos - start_pos
        elif player_bj:
            return [('win', 1.5)], pos - start_pos
        else:
            return [('lose', 1)], pos - start_pos

    action = basic_strategy(player, dealer_upcard, can_split=True, can_double=True)

    if action == 'P':
        is_aces = (player[0] == 11)
        hand1 = [player[0], shoe[pos]]; pos += 1
        hand2 = [player[1], shoe[pos]]; pos += 1

        hand1, mult1, pos = play_out_hand(shoe, pos, hand1, dealer_upcard, is_split_aces=is_aces)
        hand2, mult2, pos = play_out_hand(shoe, pos, hand2, dealer_upcard, is_split_aces=is_aces)
        hands = [(hand1, mult1), (hand2, mult2)]
    else:
        final_cards, mult, pos = play_out_hand(shoe, pos, player, dealer_upcard)
        hands = [(final_cards, mult)]

    dealer, pos = play_dealer(shoe, pos, dealer)
    dealer_total, _ = hand_value(dealer)

    results = []
    for cards, mult in hands:
        player_total, _ = hand_value(cards)
        if player_total > 21:
            results.append(('lose', mult))
        elif dealer_total > 21:
            results.append(('win', mult))
        elif player_total > dealer_total:
            results.append(('win', mult))
        elif player_total < dealer_total:
            results.append(('lose', mult))
        else:
            results.append(('push', mult))

    return results, pos - start_pos


In [ ]:
def simulate_blackjack(n_hands, n_decks=8, penetration_decks=4):
    """penetration_decks: how many decks are actually dealt before the cut card forces a reshuffle."""
    cards_per_deck = 52
    cut_card_position = penetration_decks * cards_per_deck  # 208 cards for 4 of 8 decks

    shoe = build_shoe(n_decks)
    pos = 0
    all_results = []

    for _ in range(n_hands):
        if pos > cut_card_position - 20:  # safety buffer for hands using many cards
            shoe = build_shoe(n_decks)
            pos = 0

        results, cards_used = play_hand(shoe, pos)
        pos += cards_used
        all_results.extend(results)

    return all_results

n_hands = 100_000
outcomes = simulate_blackjack(n_hands, n_decks=8, penetration_decks=4)


In [ ]:
net = 0
for result, mult in outcomes:
    if result == 'win':
        net += mult
    elif result == 'lose':
        net -= mult
    # push contributes 0

n_bets = len(outcomes)  # >= n_hands, since splits add extra bets
house_edge = -net / n_bets

wins = sum(1 for r, m in outcomes if r == 'win')
losses = sum(1 for r, m in outcomes if r == 'lose')
pushes = sum(1 for r, m in outcomes if r == 'push')

print(f"Hands dealt: {n_hands:,}")
print(f"Total bets resolved (incl. splits): {n_bets:,}")
print(f"Wins:   {wins:,} ({wins/n_bets:.4%})")
print(f"Losses: {losses:,} ({losses/n_bets:.4%})")
print(f"Pushes: {pushes:,} ({pushes/n_bets:.4%})")
print()
print(f"Actual house edge: {house_edge:.4%}  (theoretical basic strategy, 8-deck, stand-soft-17: ~0.5-0.6%)")


Hands dealt: 100,000
Total bets resolved (incl. splits): 102,512
Wins:   44,564 (43.4720%)
Losses: 49,061 (47.8588%)
Pushes: 8,887 (8.6692%)

Actual house edge: 0.5575%  (theoretical basic strategy, 8-deck, stand-soft-17: ~0.5-0.6%)
